# Entropy Simulations 
    
Below is simple simulation where adjacent cells swap position randomly.    
    
A more sophisticated version would

a) Calculate the swap probabilities based on 
$$
P = e^{\delta E / kT}
$$

This would produce diffusion physics.

b) Instead of the Shannon entropy, calculate the true combinatorial entropy
$$
S = k ln W    
$$   
where
$$    
W = \frac {N!}{N_A! N_B!}
$$


# The Gibbs Thermodynamic Potential: Why “Energy” is a Misnomer

The Gibbs free energy is defined as:

$$
G = H - T S
$$

- **H (enthalpy)**: energy associated with bonds, interactions, and PV work  
- **S (entropy)**: measure of microscopic configurational freedom  

---

## 1. Gibbs as a thermodynamic potential

Although historically called “energy,” \(G\) is **not a pure energy** like \(U\) or \(H\).  

- It is a **thermodynamic potential**, a function whose **minimum determines equilibrium** at constant temperature and pressure.  
- Calling it a potential emphasizes that it **balances energy and entropy**, rather than measuring only energy content.

---

## 2. Why entropy enters

At finite temperature \(T\):

- Each macrostate corresponds to many microscopic arrangements (microstates).  
- The more microstates a macrostate has, the **higher its entropy**, \(S\).  
- Gibbs free energy incorporates this effect:

$$
G = H - T S
$$

- Even if \(H\) is constant (like in a lattice gas with no interactions), **macrostates with higher coarse-grained mixing have higher entropy → lower G**.  
- This is why mixing occurs spontaneously at finite temperature.

---

## 3. Intuition

Think of \(G\) as a **score for each macrostate**:

$$
\text{score} = \text{energy cost} - T \times \text{number of accessible microstates}
$$

- **Lower score = more favorable**  
- Nature selects the macrostate with **minimum Gibbs potential**.  
- At \(T=0\), only energy matters; at \(T>0\), entropy drives spontaneous processes like mixing.

---

### ✅ Key Takeaway

> Gibbs free energy is a **thermodynamic potential** that combines energy and entropy.  
> It predicts equilibrium not by energy alone, but by **the trade-off between energy cost and microscopic freedom**.

# Understanding the `compute_entropy` Function

The function implements a **coarse-grained (spatial) entropy**, designed to capture how *mixed* the two gases are across space—not just how many particles of each type exist.

---

## 1. Why global entropy fails here

A naive entropy calculation uses the global fractions:

- $p_A = \frac{N_A}{N}$, $p_B = \frac{N_B}{N}$

and computes:

$$
S = -k \left(p_A \log p_A + p_B \log p_B \right)
$$

But in this simulation:
- $N_A$ and $N_B$ never change
- So $p_A = p_B = 0.5$ always

**Result:** constant entropy $S = k \log 2$, regardless of mixing.

This corresponds to a **fully coarse macroscopic description** that ignores spatial structure.

---

## 2. Coarse-graining: introducing spatial resolution

To capture mixing, the lattice is divided into **blocks** of size `block_size × block_size`.

Each block is treated as a *local subsystem* with its own probabilities:

- $p_A^{(\text{block})} = $ fraction of A particles in the block  
- $p_B^{(\text{block})} = 1 - p_A^{(\text{block})}$

This step introduces **mesoscopic resolution**—we no longer treat the system as perfectly homogeneous.

---

## 3. Local entropy inside each block

Within each block, Shannon entropy is computed:

$$
S_{\text{block}} = - \left(p_A \log p_A + p_B \log p_B \right)
$$

Interpretation:
- $S_{\text{block}} = 0$: block is pure (all A or all B)
- $S_{\text{block}} = \log 2$: block is perfectly mixed (50/50)

---

## 4. Total entropy as a sum over blocks

The total entropy is:

$$
S = \sum_{\text{blocks}} S_{\text{block}}
$$

This assumes:
- blocks are treated as **independent subsystems**
- correlations between blocks are ignored (coarse-graining approximation)

---

## 5. Physical interpretation

This entropy measures:

> How much *spatial information* about particle identity is lost when we only observe the system at the block scale.

- Initially:
  - Each block is pure → low entropy
- During mixing:
  - Blocks become heterogeneous → entropy increases
- At equilibrium:
  - Each block is ~50/50 → entropy maximal

---

## 6. Connection to statistical mechanics

This is a discrete analogue of:

- **coarse-grained Gibbs entropy**
- or **Boltzmann entropy** applied to spatial distributions

It reflects the idea behind the  
**Second Law of Thermodynamics**:

> Entropy increases because macrostates that look “mixed” correspond to vastly more microscopic configurations than ordered ones.

---

## 7. Key insight

The function does **not** measure randomness of individual particles.

Instead, it measures:

> How indistinguishable the system looks when observed with limited spatial resolution.

This explains why entropy increases—even though the microscopic dynamics (random swaps) are perfectly reversible.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# -----------------------------
# Parameters
# -----------------------------
N = 50               # lattice size
k = 1.0              # Boltzmann constant
block_size = 5       # coarse-graining block size
mix_speed = 1500     # number of swaps per frame
num_frames = 500     # number of animation frames

# -----------------------------
# Initial lattice: left A (0), right B (1)
# -----------------------------
lattice = np.zeros((N, N), dtype=int)
lattice[:, N//2:] = 1
entropy_history = []

# -----------------------------
# Vectorized coarse-grained entropy
# -----------------------------
def compute_entropy_vectorized(lattice, block_size=5):
    k = 1.0
    N = lattice.shape[0]
    assert N % block_size == 0, "Block size must divide lattice size"

    # reshape lattice into blocks
    blocks = lattice.reshape(N//block_size, block_size,
                             N//block_size, block_size)
    blocks = blocks.transpose(0, 2, 1, 3)

    # fraction of A and B in each block
    p_A = np.mean(blocks == 0, axis=(2,3))
    p_B = 1 - p_A

    # Shannon entropy per block
    S_block = np.zeros_like(p_A)
    mask_A = p_A > 0
    mask_B = p_B > 0
    S_block[mask_A] -= p_A[mask_A] * np.log(p_A[mask_A])
    S_block[mask_B] -= p_B[mask_B] * np.log(p_B[mask_B])

    # total entropy
    return k * np.sum(S_block)

# -----------------------------
# Vectorized random swaps
# -----------------------------
def random_swap_vectorized(lattice, num_swaps):
    N = lattice.shape[0]
    # random starting coordinates
    x = np.random.randint(0, N, size=num_swaps)
    y = np.random.randint(0, N, size=num_swaps)

    # random directions: 0=right,1=left,2=down,3=up
    dirs = np.random.randint(0, 4, size=num_swaps)
    dx = np.array([1,-1,0,0])[dirs]
    dy = np.array([0,0,1,-1])[dirs]

    # target coordinates with periodic BC
    x2 = (x + dx) % N
    y2 = (y + dy) % N

    # swap in batch
    lattice[x, y], lattice[x2, y2] = lattice[x2, y2], lattice[x, y].copy()

# -----------------------------
# Set up figure
# -----------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

# lattice panel
img = ax1.imshow(lattice, cmap="coolwarm", vmin=0, vmax=1)
ax1.set_title("Gas Mixing")

# entropy panel
line, = ax2.plot([], [], lw=2)
ax2.set_xlim(0, num_frames)
ax2.set_ylim(0, 1.0)
ax2.set_title("Entropy")
ax2.set_xlabel("Frame")
ax2.set_ylabel("S")

# -----------------------------
# Animation update
# -----------------------------
def update(frame):
    # perform vectorized swaps
    random_swap_vectorized(lattice, mix_speed)

    # compute vectorized coarse-grained entropy
    S = compute_entropy_vectorized(lattice, block_size)
    entropy_history.append(S)

    # update lattice image
    img.set_data(lattice)
    ax1.set_title(f"Mixing (frame {frame})")

    # update entropy plot
    line.set_data(range(len(entropy_history)), entropy_history)

    # auto-rescale y-axis
    ax2.set_ylim(0, max(entropy_history) * 1.1)

    return [img, line]

# -----------------------------
# Run animation
# -----------------------------
ani = animation.FuncAnimation(
    fig, update, frames=num_frames, interval=50, blit=False
)

HTML(ani.to_jshtml())